# Step 23 — The committor as the primary object, and a model whose class mass is fixed by construction

**Inputs.** Mount the committed **Step 19** output. GPU and Internet on. Committed run, about 5 hours.

**A. The committor at scale.** 320 of the pretrained model's reverse-SDE trajectories are stopped at 12 noise levels between 8 and 0.1. From each stopped state we run 12 independent continuations, labelled by the independent ViT (98% accurate). At every state we therefore have the model's committor $q_{\rm model}(c\mid x_\sigma)$ for all ten classes. With it we measure:

- **Committor fidelity.** For an exact model, the committor equals the data's class posterior $P(c\mid x_\sigma)$, which our noise-conditional classifier estimates. We compare the two state by state, against the multinomial noise of 12 draws. This is a model-versus-data test with no density estimate and no coordinate choice.
- **Where the mass error enters.** Averaged over the model's own states, the committor equals the final class frequency at every noise level (a martingale). Averaged over the same states, the classifier posterior shows the class mass the model's marginal already carries at that noise level. The gap to uniform traces the noise level at which over-generation appears.
- **Decisions.** The noise level at which each trajectory's superclass, class, and pairwise outcome are settled. The per-pair committor decision noise level is compared, like for like, with each pair's density transition, for the data and for the model.

**B. A model with the right mass by construction.** NVIDIA's class-conditional EDM, sampled with uniform labels, has no mass error. If commitment errors are mass errors, it should match the data in both where it commits and how lopsided the commitment is, **without** rebalancing.

In [ ]:
import os; os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import sys, subprocess, pickle, glob, math, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
from scipy.signal import fftconvolve
dev = 'cuda'; torch.manual_seed(0); rng = np.random.default_rng(0); plt.rcParams['figure.dpi'] = 120
N_COND = 4000            # conditional-model trajectories
CHUNK = 500
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']; VEH = {0, 1, 8, 9}; PAIRS = list(combinations(range(10), 2))
P19 = glob.glob('/kaggle/input/**/step19_logits.npz', recursive=True); assert P19, 'mount the Step 19 output'
Z19 = np.load(P19[0]); L_TE, W_TE, L_M0, W_M0, SIG_REC = Z19['L_TE'], Z19['W_TE'], Z19['L_M'], Z19['W_M'], Z19['SIG_REC']
SIGMA_MAX, SIGMA_MIN, RHO, STEPS = 80.0, 0.05, 7.0, 256
def edm_sched(steps=STEPS, s_max=SIGMA_MAX, s_min=SIGMA_MIN, rho=RHO):
    i = np.arange(steps); return (s_max**(1/rho) + i/(steps-1)*(s_min**(1/rho) - s_max**(1/rho)))**rho
SCHED = edm_sched(); REC = {int(np.argmin(np.abs(SCHED - s))): gi for gi, s in enumerate(SIG_REC)}
assert np.allclose(SCHED[sorted(REC)], np.sort(SIG_REC)[::-1]), 'recording grid does not match Step 19'
print('reference', L_TE.shape, 'Step 19 baseline trajectories', L_M0.shape)

# ---- classifier used for the reference (from Step 19) ----
class FiLMBlock(nn.Module):
    def __init__(self, cin, cout, emb, stride=1):
        super().__init__(); self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout); self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm2d(cout)
        self.film = nn.Linear(emb, 2*cout); self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout)) if (stride != 1 or cin != cout) else nn.Identity()
    def forward(self, x, e):
        g, b = self.film(e).chunk(2, 1); h = F.silu(self.b1(self.c1(x))); h = self.b2(self.c2(h))*(1+g[:, :, None, None]) + b[:, :, None, None]; return F.silu(h + self.sc(x))
class NoisyResNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(32, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(3, ch, 3, 1, 1)
        self.blocks = nn.ModuleList([FiLMBlock(ch, ch, emb), FiLMBlock(ch, 2*ch, emb, 2), FiLMBlock(2*ch, 2*ch, emb), FiLMBlock(2*ch, 4*ch, emb, 2), FiLMBlock(4*ch, 4*ch, emb)]); self.fc = nn.Linear(4*ch, 10)
    def forward(self, x, sigma):
        f = torch.exp(torch.arange(16, device=x.device)*-(math.log(100)/15)); t = 0.25*torch.log(sigma)[:, None]*f; e = self.emb(torch.cat([torch.sin(t), torch.cos(t)], 1)); h = self.inp(x)
        for b in self.blocks: h = b(h, e)
        return self.fc(h.mean((2, 3)))
CLF = NoisyResNet().to(dev); CLF.load_state_dict(torch.load(glob.glob('/kaggle/input/**/noisy_resnet_seed0.pt', recursive=True)[0], map_location=dev)); CLF.eval()
for p in CLF.parameters(): p.requires_grad_(False)
@torch.no_grad()
def clf_logits(x, sigma, bs=1000):
    out = []
    for xb in x.split(bs):
        with torch.autocast('cuda', dtype=torch.float16): out.append(CLF(xb, torch.full((len(xb),), float(sigma), device=dev)).float())
    return torch.cat(out)

# ---- pretrained EDM ----
if not os.path.exists('/kaggle/working/edm'): subprocess.run(['git', 'clone', '-q', 'https://github.com/NVlabs/edm.git', '/kaggle/working/edm'], check=True)
sys.path.insert(0, '/kaggle/working/edm'); import dnnlib
with dnnlib.util.open_url('https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-uncond-vp.pkl') as f: NET = pickle.load(f)['ema'].to(dev).eval()
NET.use_fp16 = True
ACTIVE = {'net': None, 'labels': None}
@torch.no_grad()
def D(x, s):
    if ACTIVE['net'] is None: return NET(x, torch.full((len(x),), float(s), device=dev)).float()
    return ACTIVE['net'](x, torch.full((len(x),), float(s), device=dev), ACTIVE['labels'][:len(x)]).float()
# reference class frequencies (the classifier's labels on test data) = mass-correction target
F_REF = np.bincount(W_TE.argmax(1), minlength=10)/len(W_TE); F_BASE19 = np.bincount(W_M0.argmax(1), minlength=10)/len(W_M0)
print('classifier label frequency on test:', np.round(F_REF, 3)); print('Step 19 generated frequency:     ', np.round(F_BASE19, 3))

In [ ]:
def guide_grad(x, s, logr):
    """Exact class-reweighting tilt: grad_x [ logsumexp(l + log r) - logsumexp(l) ] with the noise-conditional classifier."""
    with torch.enable_grad():
        xg = x.detach().requires_grad_(True); l = CLF(xg, torch.full((len(xg),), float(s), device=dev)).float()
        obj = (torch.logsumexp(l + logr, 1) - torch.logsumexp(l, 1)).sum(); return torch.autograd.grad(obj, xg)[0]
@torch.no_grad()
def run_sde(n, seed, logr=None, window=None, lam=1.0, keep_states_at=None):
    """Reverse SDE (Euler-Maruyama, 256 steps) as in Step 19. Returns logits at the recorded levels, terminal weights, final denoised images
    (uint8), and optionally the states at given step indices (for branching)."""
    g = torch.Generator(device=dev).manual_seed(seed); x = torch.randn(n, 3, 32, 32, generator=g, device=dev)*SCHED[0]
    Lrec = np.zeros((len(SIG_REC), n, 10), np.float16); states = {}
    lr = None if logr is None else torch.as_tensor(logr, dtype=torch.float32, device=dev)
    for i in range(STEPS):
        s0 = float(SCHED[i]); s1 = float(SCHED[i+1]) if i+1 < STEPS else SIGMA_MIN
        if i in REC: Lrec[REC[i]] = clf_logits(x, s0).cpu().numpy().astype(np.float16)
        if keep_states_at is not None and i in keep_states_at: states[i] = x.clone()
        if s1 >= s0: continue
        d = D(x, s0); sc = lam*(d - x)/s0**2
        if lr is not None and (window is None or window[0] <= s0 <= window[1]): sc = sc + guide_grad(x, s0, lr)
        x = x + (s1 - s0)*(-2*s0*sc) + math.sqrt(2*s0*(s0 - s1))*torch.randn(x.shape, generator=g, device=dev)
    W = F.softmax(clf_logits(x, SIGMA_MIN), 1).cpu().numpy(); x0 = D(x, SIGMA_MIN).clamp(-1, 1)
    img = ((x0 + 1)*127.5).round().to(torch.uint8).cpu().numpy()
    return Lrec, W, img, states
def run_condition(name, n, seed0, **kw):
    t0 = time.time(); Ls, Ws, Is = [], [], []
    for c in range(0, n, CHUNK):
        L_, W_, I_, _ = run_sde(min(CHUNK, n - c), seed0 + c, **kw); Ls.append(L_); Ws.append(W_); Is.append(I_)
        if c == 0: print(f"  {name}: first chunk {time.time()-t0:.0f}s -> estimated {(time.time()-t0)*math.ceil(n/CHUNK)/60:.0f} min")
    L_, W_, I_ = np.concatenate(Ls, 1), np.concatenate(Ws), np.concatenate(Is)
    np.savez_compressed(f'/kaggle/working/step21_{name}.npz', L=L_, W=W_, IMG=I_); print(f"  {name}: {len(W_)} trajectories ({(time.time()-t0)/60:.0f} min)"); return L_, W_, I_

def to_float(img): return torch.from_numpy(img).float().div(127.5).sub(1).to(dev)
@torch.no_grad()
def our_labels(img): return np.concatenate([clf_logits(to_float(img[i:i+1000]) + SIGMA_MIN*torch.randn(len(img[i:i+1000]), 3, 32, 32, device=dev), SIGMA_MIN).argmax(1).cpu().numpy() for i in range(0, len(img), 1000)])
# independent strong classifier: ViT fine-tuned on CIFAR-10 (HuggingFace)
VIT = None
try:
    from transformers import AutoModelForImageClassification
    VIT = AutoModelForImageClassification.from_pretrained('aaraki/vit-base-patch16-224-in21k-finetuned-cifar10').to(dev).eval().half()
    names = [VIT.config.id2label[i].lower() for i in range(10)]; VMAP = np.array([CLASSES.index(n) if n in CLASSES else i for i, n in enumerate(names)]); print('ViT labels:', names)
except Exception as e: print('ViT unavailable:', e)
@torch.no_grad()
def vit_labels(img, bs=128):
    out = []
    for i in range(0, len(img), bs):
        x = to_float(img[i:i+bs]); x = F.interpolate(x, size=224, mode='bilinear', align_corners=False)          # [-1,1] == (x-0.5)/0.5 normalisation
        out.append(VIT(pixel_values=x.half()).logits.argmax(1).cpu().numpy())
    return VMAP[np.concatenate(out)]
dst = datasets.CIFAR10('/kaggle/working/cifar', train=False, download=True); TEST_IMG = np.asarray(dst.data).transpose(0, 3, 1, 2); YTE = np.asarray(dst.targets)
if VIT is not None: print(f"ViT test accuracy (2000 images): {np.mean(vit_labels(TEST_IMG[:2000]) == YTE[:2000]):.3f}")
print(f"our classifier test accuracy (sigma_min): {np.mean(our_labels(TEST_IMG[:2000]) == YTE[:2000]):.3f}")
# FID (pytorch-fid Inception) against the 50k training images

dst = datasets.CIFAR10('/kaggle/working/cifar', train=False, download=True); TEST_IMG = np.asarray(dst.data).transpose(0, 3, 1, 2); YTE = np.asarray(dst.targets)
print(f"ViT test accuracy (2000): {np.mean(vit_labels(TEST_IMG[:2000]) == YTE[:2000]):.3f}")

## A. Branching: committors for every state

In [ ]:
M_PAR, K_BR = 320, 12
LEV = sorted({int(np.argmin(np.abs(SCHED - s))) for s in np.geomspace(8.0, 0.1, 12)}); LEV_SIG = SCHED[LEV]
print('branching levels:', np.round(LEV_SIG, 3), '| remaining steps:', [STEPS - i for i in LEV])
t0 = time.time(); ST, PAR_IMG = {i: [] for i in LEV}, []
for c0 in range(0, M_PAR, 160):
    _, _, img, st = run_sde(min(160, M_PAR - c0), 31000 + c0, keep_states_at=set(LEV))
    PAR_IMG.append(img); [ST[i].append(st[i]) for i in LEV]
ST = {i: torch.cat(v) for i, v in ST.items()}; PAR_IMG = np.concatenate(PAR_IMG); PAR = vit_labels(PAR_IMG)
Q_CLF = np.stack([F.softmax(clf_logits(ST[i], float(SCHED[i])), 1).cpu().numpy() for i in LEV])          # [L, M, 10] data-posterior estimate at each state
print(f"parents: {M_PAR} ({time.time()-t0:.0f}s); parent class frequencies (ViT):", np.round(np.bincount(PAR, minlength=10)/M_PAR, 3))
@torch.no_grad()
def continue_img(x, i_start, seed):
    g = torch.Generator(device=dev).manual_seed(seed)
    for i in range(i_start, STEPS):
        s0 = float(SCHED[i]); s1 = float(SCHED[i+1]) if i+1 < STEPS else SIGMA_MIN
        if s1 >= s0: continue
        sc = (D(x, s0) - x)/s0**2; x = x + (s1 - s0)*(-2*s0*sc) + math.sqrt(2*s0*(s0 - s1))*torch.randn(x.shape, generator=g, device=dev)
    return ((D(x, SIGMA_MIN).clamp(-1, 1) + 1)*127.5).round().to(torch.uint8).cpu().numpy()
END = np.zeros((len(LEV), M_PAR, K_BR), int)
for li, i0 in enumerate(LEV):
    xs = ST[i0].repeat_interleave(K_BR, 0); labs = []
    for c0 in range(0, len(xs), CHUNK): labs.append(vit_labels(continue_img(xs[c0:c0+CHUNK], i0, 50000 + 1000*li + c0)))
    END[li] = np.concatenate(labs).reshape(M_PAR, K_BR); print(f"  level sigma={LEV_SIG[li]:.3f} done ({(time.time()-t0)/60:.0f} min)")
np.savez_compressed('/kaggle/working/step23_committor.npz', END=END, PAR=PAR, Q_CLF=Q_CLF, LEV_SIG=LEV_SIG)
Q_MOD = np.stack([np.stack([np.bincount(END[l, m], minlength=10)/K_BR for m in range(M_PAR)]) for l in range(len(LEV))])     # [L, M, 10]

In [ ]:
# ---- committor fidelity: model committor vs data posterior, against 12-draw multinomial noise ----
rs = np.random.default_rng(1); print(f"{'sigma':>7} {'TV(model committor, data posterior)':>36} {'expected from 12 draws':>24} {'excess':>8}")
for l, s in enumerate(LEV_SIG):
    tv = 0.5*np.abs(Q_MOD[l] - Q_CLF[l]).sum(1).mean()
    sim = np.mean([0.5*np.abs(np.bincount(rs.choice(10, K_BR, p=q/q.sum()), minlength=10)/K_BR - q).sum() for q in Q_CLF[l]])
    print(f"{s:7.3f} {tv:36.3f} {sim:24.3f} {tv - sim:+8.3f}")
# ---- where the mass error enters: mean committor (= final frequency) vs mean data posterior over the model's own states ----
print(f"\n{'sigma':>7} " + ' '.join(f"{c[:5]:>6}" for c in CLASSES) + "   (mean data posterior over the model's states at this noise level)")
for l, s in enumerate(LEV_SIG): print(f"{s:7.3f} " + ' '.join(f"{v:6.3f}" for v in Q_CLF[l].mean(0)))
print(f"{'final':>7} " + ' '.join(f"{v:6.3f}" for v in np.bincount(PAR, minlength=10)/M_PAR) + "   (final ViT frequencies of the same trajectories)")
dev_ = [np.mean(np.abs(np.log(Q_CLF[l].mean(0)/0.1))) for l in range(len(LEV))]
print('mean |log(posterior mass / uniform)| by noise level:', ' '.join(f"{s:.2f}:{d:.3f}" for s, d in zip(LEV_SIG, dev_)))

# ---- decisions: superclass, class, and per pair ----
isv = lambda x: np.isin(x, list(VEH)); THR = 11/12
def decision(frac):
    out = np.full(frac.shape[1], np.nan)
    for j in range(frac.shape[1]):
        dec = frac[:, j] >= THR; tail = np.flip(np.cumprod(np.flip(dec))).astype(bool)
        if tail.any(): out[j] = LEV_SIG[np.argmax(tail)]
    return out
same_sup = (isv(END) == isv(PAR)[None, :, None]).mean(2); same_cls = (END == PAR[None, :, None]).mean(2)
dsup, dcls = decision(same_sup), decision(same_cls)
print(f"\ndecided (>= 11 of 12 continuations agree from this level on): superclass median {np.nanmedian(dsup):.2f} (IQR {np.nanpercentile(dsup,25):.2f}-{np.nanpercentile(dsup,75):.2f}); "
      f"class median {np.nanmedian(dcls):.2f} (IQR {np.nanpercentile(dcls,25):.2f}-{np.nanpercentile(dcls,75):.2f}); vehicles' class {np.nanmedian(dcls[isv(PAR)]):.2f}, animals' class {np.nanmedian(dcls[~isv(PAR)]):.2f}")
def pair_decision(a, b, min_par=15, min_n=6):
    js = np.where(np.isin(PAR, [a, b]))[0]
    if len(js) < min_par: return None
    C_ = []
    for l in range(len(LEV)):
        committed = []
        for j in js:
            na, nb_ = np.sum(END[l, j] == a), np.sum(END[l, j] == b)
            if na + nb_ >= min_n: committed.append((na if PAR[j] == a else nb_)/(na + nb_) >= 0.9)
        C_.append(np.mean(committed) if len(committed) >= min_par//2 else np.nan)
    C_ = np.array(C_); ok = np.isfinite(C_)
    if not ok.any() or np.nanmax(C_) < 0.5: return None
    for l in range(len(LEV)):                                # highest noise level from which at least half are committed (log-interpolated)
        if ok[l] and C_[l] >= 0.5:
            if l > 0 and ok[l-1]: w = (0.5 - C_[l-1])/(C_[l] - C_[l-1]); return float(np.exp(np.log(LEV_SIG[l-1]) + w*(np.log(LEV_SIG[l]) - np.log(LEV_SIG[l-1]))))
            return float(LEV_SIG[l])
    return None
DEC = {p: pair_decision(*p) for p in PAIRS}; DEC = {p: v for p, v in DEC.items() if v}
print(f"per-pair committor decisions resolved for {len(DEC)} pairs")

In [ ]:
G = np.linspace(-4.0, 4.0, 4001); DGG = G[1]-G[0]
def wkde_score(m, w):
    m = np.clip(m, -8, 8); w = np.asarray(w, dtype=np.float64); neff = w.sum()**2/np.maximum((w**2).sum(), 1e-12)
    mu = np.average(m, weights=w); sd = np.sqrt(np.average((m-mu)**2, weights=w)); bw = max(1.5*1.06*sd*neff**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(G-DGG/2, G[-1]+DGG/2), weights=w, density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum()
    p = np.maximum(fftconvolve(h, k, 'same'), 1e-300); return np.gradient(np.log(p), G), p
def _merge(pts, merge=0.03):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary(sm, p, lo=-3.5, hi=3.5, prominence=0.05):
    keep = (G > lo) & (G < hi) & (p > 1e-3*p.max()); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(G[i]-sm[i]*DGG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(G[i]), 'max' if sm[i] > 0 else 'min') for i in idx])
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DGG)]); lp = lambda x: float(np.interp(x, G, logp)); ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def cusp_h(sm, p, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(G) < window) & (p > 0); c3, c2, c1, c0 = np.polyfit(G[ok], sm[ok], 3, w=np.sqrt(p[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return c0 + c1*s + c2*s**2 + c3*s**3
def std_stats(z, wa, wb):
    mua, mub = np.average(z, weights=wa), np.average(z, weights=wb); half = 0.5*(mua-mub); sda = np.sqrt(np.average((z-mua)**2, weights=wa)); sdb = np.sqrt(np.average((z-mub)**2, weights=wb))
    return 0.5*(mua+mub), half, abs(half)/(0.5*(sda+sdb))
STD = {(a, b): [std_stats(L_TE[g, :, a].astype(np.float64) - L_TE[g, :, b], W_TE[:, a] + 1e-12, W_TE[:, b] + 1e-12) for g in range(len(SIG_REC))] for a, b in PAIRS}
def pair_sh(L, W, a, b, idx, balance=False):
    """(sigma_bif, h) along the pair log-odds, standardised by the reference, weighted by terminal class weights."""
    Wa = W[idx, a].astype(np.float64); Wb = W[idx, b].astype(np.float64)
    if balance: tot = 0.5*(Wa.sum() + Wb.sum()); Wa = Wa/Wa.sum()*tot; Wb = Wb/Wb.sum()*tot
    prev = None
    for g in range(len(SIG_REC)):
        mid, half, sep = STD[(a, b)][g]
        if sep < 0.3: prev = 1; continue
        z = L[g, idx, a].astype(np.float64) - L[g, idx, b]; sm, p = wkde_score((z - mid)/half, Wa + Wb); nm = n_max(stationary(sm, p))
        if nm >= 2 and prev is not None: return float(SIG_REC[g]), cusp_h(sm, p)
        if nm == 1: prev = 1
    return None, None
def evaluate(L, W, name, NB=40):
    n = len(W); ir = rng.choice(len(W_TE), size=n, replace=False); allm = np.arange(n); rows = []
    fm = np.bincount(W.argmax(1), minlength=10)/n; over = np.log(fm/F_REF)
    for a, b in PAIRS:
        sm_, hm = pair_sh(L, W, a, b, allm); sr, hr = pair_sh(L_TE, W_TE, a, b, ir); _, hmb = pair_sh(L, W, a, b, allm, balance=True)
        if sm_ is None or sr is None: continue
        rat = []
        for _ in range(NB):
            s1, _ = pair_sh(L, W, a, b, rng.integers(n, size=n)); s2, _ = pair_sh(L_TE, W_TE, a, b, rng.choice(len(W_TE), size=n))
            if s1 and s2: rat.append(s1/s2)
        lo, hi = np.percentile(rat, [2.5, 97.5]) if len(rat) > 10 else (np.nan, np.nan)
        rows.append(dict(pair=(a, b), sm=sm_, sr=sr, hm=hm, hr=hr, hmb=hmb, lo=lo, hi=hi, dover=over[a]-over[b]))
    R = np.array([r['sm']/r['sr'] for r in rows]); hm = np.array([r['hm'] for r in rows]); hr = np.array([r['hr'] for r in rows]); hmb = np.array([r['hmb'] for r in rows]); dov = np.array([r['dover'] for r in rows])
    lo = np.array([r['lo'] for r in rows]); hi = np.array([r['hi'] for r in rows])
    print(f"{name:>20}: pairs {len(rows)}; sigma_bif ratio median {np.median(R):.2f} (CI>1: {(lo>1).sum()}, CI<1: {(hi<1).sum()}); corr h model-data raw {np.corrcoef(hm, hr)[0,1]:.2f}, rebalanced {np.corrcoef(hmb, hr)[0,1]:.2f}; "
          f"corr h model vs over-generation {np.corrcoef(hm, dov)[0,1]:.2f}; mean |log over-generation| {np.mean(np.abs(over)):.3f}")
    return {r['pair']: r for r in rows}, fm

# ---- per-pair: committor decision vs density transition (data and model, Step 19 trajectories) ----
L_M0n, W_M0n = L_M0, W_M0; n = len(W_M0n); rows = []
for p, sd in DEC.items():
    a, b = p; ir = rng.choice(len(W_TE), size=n, replace=False); s_dat, _ = pair_sh(L_TE, W_TE, a, b, ir); s_mod, _ = pair_sh(L_M0n, W_M0n, a, b, np.arange(n))
    if s_dat and s_mod: rows.append((p, sd, s_dat, s_mod))
from scipy.stats import spearmanr
sd_ = np.array([r[1] for r in rows]); sdat = np.array([r[2] for r in rows]); smod = np.array([r[3] for r in rows])
print(f"\n{len(rows)} pairs: Spearman(committor decision, data sigma_bif) = {spearmanr(sd_, sdat).correlation:.2f}; Spearman(committor decision, model sigma_bif) = {spearmanr(sd_, smod).correlation:.2f}")
print(f"   decision / sigma_bif(model): median {np.median(sd_/smod):.2f} (IQR {np.percentile(sd_/smod,25):.2f}-{np.percentile(sd_/smod,75):.2f});  decision / sigma_bif(data): median {np.median(sd_/sdat):.2f}")
kd = lambda p: 'va' if ((p[0] in VEH) != (p[1] in VEH)) else ('vv' if p[0] in VEH else 'aa')
for k in ['va', 'vv', 'aa']:
    j = [i for i, r in enumerate(rows) if kd(r[0]) == k]
    if j: print(f"   {k}: median committor decision {np.median(sd_[j]):.2f}, median model sigma_bif {np.median(smod[j]):.2f}, median data sigma_bif {np.median(sdat[j]):.2f} (n={len(j)})")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
ax[0].plot(LEV_SIG, same_sup.mean(1), 'o-', label='same superclass'); ax[0].plot(LEV_SIG, same_cls.mean(1), 's-', label='same class'); ax[0].set_xscale('log'); ax[0].invert_xaxis(); ax[0].set_xlabel('noise level of the branching state'); ax[0].set_ylabel('fraction of continuations agreeing with the trajectory'); ax[0].legend(fontsize=7)
col = ['C0' if kd(r[0]) == 'va' else 'C1' if kd(r[0]) == 'vv' else 'C2' for r in rows]
ax[1].scatter(smod, sd_, c=col, s=16); lim = [0.1, 10]; ax[1].plot(lim, lim, 'k--', lw=0.6); ax[1].set_xscale('log'); ax[1].set_yscale('log'); ax[1].set_xlabel('density transition sigma_bif (model)'); ax[1].set_ylabel('committor decision (model)'); ax[1].set_title('decision vs density transition, per pair')
for l in range(len(LEV)): ax[2].scatter(np.full(10, LEV_SIG[l]), Q_CLF[l].mean(0)/0.1, c=range(10), cmap='tab10', s=12)
ax[2].axhline(1, color='k', lw=0.5); ax[2].set_xscale('log'); ax[2].invert_xaxis(); ax[2].set_xlabel('noise level'); ax[2].set_ylabel('posterior class mass / uniform'); ax[2].set_title("where the model's mass error appears")
plt.tight_layout(); plt.show()

## B. The class-conditional EDM with uniform labels: mass is right by construction

In [ ]:
with dnnlib.util.open_url('https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-cond-vp.pkl') as f: NET_C = pickle.load(f)['ema'].to(dev).eval()
NET_C.use_fp16 = True; print('loaded conditional EDM; label dim', NET_C.label_dim)
LC, WC, IC = [], [], []; t0 = time.time()
for c0 in range(0, N_COND, CHUNK):
    n_ = min(CHUNK, N_COND - c0); lab = torch.arange(c0, c0 + n_, device=dev) % 10
    ACTIVE['net'] = NET_C; ACTIVE['labels'] = F.one_hot(lab, 10).float()
    L_, W_, I_, _ = run_sde(n_, 70000 + c0); LC.append(L_); WC.append(W_); IC.append(I_)
    if c0 == 0: print(f"first chunk {time.time()-t0:.0f}s")
ACTIVE['net'] = None; LC, WC, IC = np.concatenate(LC, 1), np.concatenate(WC), np.concatenate(IC)
np.savez_compressed('/kaggle/working/step23_conditional.npz', L=LC, W=WC, IMG=IC)
fv = np.bincount(vit_labels(IC), minlength=10)/len(IC); print('conditional model, ViT class frequencies:', np.round(fv, 3), f"(mean |log f/0.1| = {np.mean(np.abs(np.log(fv/0.1))):.3f})")
evc = evaluate(LC, WC, 'conditional EDM, uniform labels')
evu = evaluate(L_M0[:, :N_COND], W_M0[:N_COND], 'unconditional EDM (Step 19 subsample)')
for name, (ev, _) in [('conditional', evc), ('unconditional', evu)]:
    P_ = list(ev); lr = np.array([np.log(ev[p]['sm']/ev[p]['sr']) for p in P_])
    print(f"   {name:>13}: SD of log(sigma_model/sigma_data) = {lr.std():.3f}; corr(raw h model, h data) = {np.corrcoef([ev[p]['hm'] for p in P_], [ev[p]['hr'] for p in P_])[0,1]:.2f}")

## What to report back
- **A.**
  - The fidelity table and the where-mass-enters table.
  - The decision line, the per-pair count, the Spearman lines, the category lines, and the three panels.
- **B.** The conditional model's ViT frequencies, the two `evaluate` lines, and the two SD/correlation lines.

The key comparison: the conditional model's **raw** asymmetry agreement and location scatter, against the unconditional model's raw and rebalanced values.